# V2R mega-tree pipeline

Intact-only backbone -> align -> FastTree (quick look) -> IQ-TREE (final ML tree).
Figures live in `v2r-tree-plotting.ipynb` (chromosome labels, species colours).

Run shell steps with the `general` env's own `python3` kernel (has mafft/trimal/FastTree;
IQ-TREE lives in `v2r_phylo`, activated inside each `%%bash` cell as needed).

In [ ]:
import re
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from Bio import Phylo
from IPython.display import Image, display
import sys

In [1]:
# project paths
PROJ = "/scratchdata1/users/a1864358/sanders_lab/phylogenomics/v2r_tree"
DATA = f"{PROJ}/workspace/data"
RES = f"{PROJ}/results"
LOGS = f"{PROJ}/workspace/logs"


#### Assemble the intact-only backbone

Pull `class=intact` records per taxon, re-header to `<sptag>_V2R####`, concat.

In [ ]:

CUR = "/hpcfs/users/a1864358/sanders_lab/asm/files/annotation/eviann_results/hmaj/curation"
V2R12 = "/scratchdata1/users/a1864358/sanders_lab/phylogenomics/orthofinder/v2r_all12"

SOURCES = [
    ("hmaj",    f"{V2R12}/hmaj.fa"),
    ("hcy",     f"{V2R12}/hcy.fa"),
    ("hcure",   f"{V2R12}/hcure.fa"),
    ("hcurw",   f"{V2R12}/hcurw.fa"),
    ("horn",    f"{V2R12}/horn.fa"),
    ("nscut",   f"{CUR}/results/nscut_denovo/v2r_nscut_denovo.fasta"),
    ("ptext",   f"{CUR}/results/ptext_denovo/v2r_ptext_denovo.fasta"),
    ("nhel",    f"{CUR}/results/nhel/v2r_nhel.fasta"),
    ("tele",    f"{CUR}/results/tele/v2r_tele.fasta"),
    ("vberus",  f"{CUR}/results/vberus/v2r_vberus.fasta"),
    ("caspera", f"{CUR}/results/caspera/v2r_caspera.fasta"),
    ("vkomodo", f"{CUR}/results/vkomodo/v2r_vkomodo.fasta"),
]

bb_path = f"{DATA}/backbone.faa"
map_path = f"{DATA}/gene_species_map.tsv"

if os.path.exists(bb_path):
    print(f"already built: {bb_path} (delete it to rebuild)")
else:
    total = 0
    with open(bb_path, "w") as bb, open(map_path, "w") as mp:
        for tag, path in SOURCES:
            n, keep = 0, False
            for line in open(path):
                if line.startswith(">"):
                    keep = "class=intact" in line
                    if keep:
                        n += 1
                        leaf = f"{tag}_V2R{n:04d}"
                        bb.write(f">{leaf}\n")
                        mp.write(f"{leaf}\t{tag}\n")
                elif keep:
                    bb.write(line if line.endswith("\n") else line + "\n")
            total += n
            print(f"  {tag:<8} {n} intact")
    print(f"total backbone tips: {total}")


already built: /scratchdata1/users/a1864358/sanders_lab/phylogenomics/v2r_tree/workspace/data/backbone.faa (delete it to rebuild)


#### Align + domain-trim

MAFFT `--auto`, then `trimal -automated1` with a 200-col floor fallback (`-gt 0.1`). SLURM job, ~12h ceiling for 2400+ seqs.

In [3]:
%%bash
PROJ=/scratchdata1/users/a1864358/sanders_lab/phylogenomics/v2r_tree
D=$PROJ/workspace/data
if [ -s "$D/backbone.trim.aln" ]; then
  echo "already aligned: $D/backbone.trim.aln"
  exit 0
fi

cat > /tmp/20_align_trim.sh <<'SBATCH'
#!/bin/bash
#SBATCH -J v2r_aln
#SBATCH -p icelake
#SBATCH -c 32
#SBATCH --mem=64G
#SBATCH -t 12:00:00
#SBATCH -o /scratchdata1/users/a1864358/sanders_lab/phylogenomics/v2r_tree/workspace/logs/20_aln_%j.out
set -euo pipefail
D=/scratchdata1/users/a1864358/sanders_lab/phylogenomics/v2r_tree/workspace/data
source /hpcfs/users/a1864358/miniconda/miniconda3/etc/profile.d/conda.sh
conda activate general
T=${SLURM_CPUS_PER_TASK:-32}

mafft --auto --anysymbol --thread "$T" "$D/backbone.faa" > "$D/backbone.aln"
trimal -in "$D/backbone.aln" -out "$D/backbone.trim.aln" -automated1
COLS=$(awk '/^>/{if(s)exit;s=1;next}{w+=length($0)}END{print w}' "$D/backbone.trim.aln")
if [ "${COLS:-0}" -lt 200 ]; then
  echo "WARN: over-trimmed to $COLS cols, falling back to -gt 0.1"
  trimal -in "$D/backbone.aln" -out "$D/backbone.trim.aln" -gt 0.1
fi
echo DONE
SBATCH

sbatch --wait /tmp/20_align_trim.sh


already aligned: /scratchdata1/users/a1864358/sanders_lab/phylogenomics/v2r_tree/workspace/data/backbone.trim.aln


#### Build the tree

FastTree (minutes, quick sanity check) then IQ-TREE ML with 1000 UFBoot (hours-days, the real result).
Skip-if-exists guards -- never re-runs a finished multi-hour tree by accident.

In [4]:
%%bash
PROJ=/scratchdata1/users/a1864358/sanders_lab/phylogenomics/v2r_tree
D=$PROJ/workspace/data; R=$PROJ/results

if [ ! -s "$R/mega.fasttree.nwk" ]; then
  cat > /tmp/30a_fasttree.sh <<'SBATCH'
#!/bin/bash
#SBATCH -J v2r_ft
#SBATCH -p icelake
#SBATCH -c 8
#SBATCH --mem=32G
#SBATCH -t 04:00:00
#SBATCH -o /scratchdata1/users/a1864358/sanders_lab/phylogenomics/v2r_tree/workspace/logs/30a_ft_%j.out
set -euo pipefail
source /hpcfs/users/a1864358/miniconda/miniconda3/etc/profile.d/conda.sh
conda activate general
export OMP_NUM_THREADS=${SLURM_CPUS_PER_TASK:-8}
D=/scratchdata1/users/a1864358/sanders_lab/phylogenomics/v2r_tree/workspace/data
R=/scratchdata1/users/a1864358/sanders_lab/phylogenomics/v2r_tree/results
FastTreeMP -lg -gamma < "$D/backbone.trim.aln" > "$R/mega.fasttree.nwk"
SBATCH
  sbatch --wait /tmp/30a_fasttree.sh
else
  echo "already have: $R/mega.fasttree.nwk"
fi






if [ ! -s "$R/mega.ml.treefile" ]; then
  cat > /tmp/30b_iqtree.sh <<'SBATCH'
#!/bin/bash
#SBATCH -J v2r_iq
#SBATCH -p icelake
#SBATCH -c 32
#SBATCH --mem=64G
#SBATCH -t 3-00:00:00
#SBATCH -o /scratchdata1/users/a1864358/sanders_lab/phylogenomics/v2r_tree/workspace/logs/30b_iq_%j.out
set -euo pipefail
source /hpcfs/users/a1864358/miniconda/miniconda3/etc/profile.d/conda.sh
conda activate v2r_phylo
D=/scratchdata1/users/a1864358/sanders_lab/phylogenomics/v2r_tree/workspace/data
R=/scratchdata1/users/a1864358/sanders_lab/phylogenomics/v2r_tree/results
T=${SLURM_CPUS_PER_TASK:-32}
iqtree2 -s "$D/backbone.trim.aln" --prefix "$R/mega.ml" -m LG+F+G4 -B 1000 --bnni -T "$T"
SBATCH
  sbatch --wait /tmp/30b_iqtree.sh
else
  echo "already have: $R/mega.ml.treefile"
fi


already have: /scratchdata1/users/a1864358/sanders_lab/phylogenomics/v2r_tree/results/mega.fasttree.nwk
already have: /scratchdata1/users/a1864358/sanders_lab/phylogenomics/v2r_tree/results/mega.ml.treefile


#### Figure

See `v2r-tree-plotting.ipynb` for the radial figure (chromosome labels, species colours, pruning).

For a quick look:

In [10]:

species_colors = {
    "hmaj": "red",
    "hcy": "blue",
    "hcure": "green",
    "hcurw": "orange",
    "horn": "purple",
    "nscut": "pink",
    "ptext": "brown",
    "nhel": "gray",
    "tele": "cyan",
    "vberus": "magenta",
    "caspera": "yellow",
    "vkomodo": "lime",
}

# tip name starts with the species code. this order is the legend order
def get_species(label):
    for species in species_colors:
        if label and label.startswith(species):
            return species
    return None

# chrom=ch2 -> 2. chrom=ACCESSION:start-end -> the accession
def chrom_token(header):
    hit = re.search(r"chrom=([^:\s|]+)", header or "")
    raw = hit.group(1) if hit else ""
    if raw.startswith("ch") and raw[2:].replace("chr", "").isdigit():
        return raw[2:]
    if raw.startswith("chr") and raw[3:].isdigit():
        return raw[3:]
    return raw or "unplaced"

# same walk as the backbone so tip ids match the tree
tip_chrom = {}
for tag, path in SOURCES:
    n = 0
    for line in open(path):
        if line.startswith(">") and "class=intact" in line:
            n += 1
            tip_chrom[f"{tag}_V2R{n:04d}"] = chrom_token(line)

t = Phylo.read(f"{RES}/mega.ml.treefile", "newick")

# y = leaf rank in the newick. x = branch length from the root
def set_y(clade, state):
    if clade.is_terminal():
        clade.y = state[0]
        state[0] += 1
        return
    for child in clade.clades:
        set_y(child, state)
    clade.y = sum(child.y for child in clade.clades) / len(clade.clades)

def set_x(clade, x):
    clade.x = x
    for child in clade.clades:
        set_x(child, x + (child.branch_length or 0.0))

def draw_clade(ax, clade):
    for child in clade.clades:
        ax.plot([clade.x, clade.x], [clade.y, child.y], color="#444444", lw=0.35, solid_capstyle="round")
        ax.plot([clade.x, child.x], [child.y, child.y], color="#444444", lw=0.35, solid_capstyle="round")
        draw_clade(ax, child)

set_y(t.root, [0])
set_x(t.root, 0.0)
tips = sorted(t.get_terminals(), key=lambda c: c.y)

# add another strip with ("name", lambda tip: value)
columns = [
    ("species", lambda tip: get_species(tip.name) or "?"),
    ("chromosome", lambda tip: tip_chrom.get(tip.name, "unplaced")),
]

# one colour per label, spaced along the colormap
def palette_for(ordered, cmap_name):
    cmap = plt.get_cmap(cmap_name)
    n = len(ordered)
    return {v: cmap(i / max(n - 1, 1)) for i, v in enumerate(ordered)}

species_order = [s for s in species_colors if any(get_species(tip.name) == s for tip in tips)]
chrom_keys = sorted(
    {tip_chrom.get(tip.name, "unplaced") for tip in tips},
    key=lambda s: (0, int(s)) if str(s).isdigit() else (1, str(s)),
)
species_pal = palette_for(species_order, "Spectral")
chrom_pal = palette_for(chrom_keys, "RdYlBu")
palettes = [species_pal, chrom_pal]

fig, (ax_t, ax_h) = plt.subplots(
    1, 2, figsize=(60, 70), sharey=True,
    gridspec_kw={"width_ratios": [3.2, 1], "wspace": 0.04},
)
draw_clade(ax_t, t.root)
x_max = max(tip.x for tip in tips)
ax_t.set_xlim(-0.01 * x_max, x_max * 1.02)
ax_t.set_ylim(-0.5, len(tips) - 0.5)
ax_t.set_xlabel("substitutions per site")
ax_t.set_yticks([])
for spine in ax_t.spines.values():
    spine.set_visible(False)

# strip width and the gap between the two columns
strip_w = 0.86
gap = 0.04
for j, ((name, getter), pal) in enumerate(zip(columns, palettes)):
    x = j * (strip_w + gap)
    for tip in tips:
        value = getter(tip)
        ax_h.add_patch(Rectangle(
            (x, tip.y - 0.5), strip_w, 1,
            facecolor=pal.get(value, "#dddddd"), edgecolor="none", linewidth=0,
        ))
    ax_h.text(x + strip_w / 2, len(tips) - 0.2, name, rotation=90, ha="center", va="bottom", fontsize=8)
ax_h.set_xlim(-0.02, 2 * strip_w + gap + 0.02)
ax_h.set_xticks([])
ax_h.set_yticks([])
for spine in ax_h.spines.values():
    spine.set_visible(False)

named = [s for s in chrom_keys if str(s).isdigit() or str(s) in {"Z", "W", "chZ", "chW"}]
species_handles = [Line2D([], [], marker="s", linestyle="none", color=species_pal[s], label=s, markersize=7) for s in species_order]
chrom_handles = [Line2D([], [], marker="s", linestyle="none", color=chrom_pal[s], label=s, markersize=6) for s in named]

# both legends sit above the strips
leg1 = ax_h.legend(
    handles=species_handles, title="species",
    loc="upper center", bbox_to_anchor=(0.5, 1.12),
    frameon=False, fontsize=22, title_fontsize=24, ncol=len(species_handles)
)
ax_h.add_artist(leg1)
ax_h.legend(
    handles=chrom_handles,
    title=f"chromosome\n({len(chrom_keys) - len(named)} scaffolds unlisted)",
    loc="upper center", bbox_to_anchor=(0.5, 1.025),
    frameon=False, fontsize=17, title_fontsize=19, ncol=min(12, len(chrom_handles))
)

out = f"{RES}/quicklook.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"wrote {out}  tips={len(tips)}  chromosomes={len(chrom_pal)}")
display(Image(filename=out))


wrote /scratchdata1/users/a1864358/sanders_lab/phylogenomics/v2r_tree/results/quicklook.png  tips=2436  chromosomes=207


In [7]:
# to add another column to heatmap
"""
columns = [
    ("species", lambda tip: get_species(tip.name) or "?"),
    ("chromosome", lambda tip: tip_chrom.get(tip.name, "unplaced")),
    ("metric", lambda tip: some_dict[tip.name]),
]
"""

'\ncolumns = [\n    ("species", lambda tip: get_species(tip.name) or "?"),\n    ("chromosome", lambda tip: tip_chrom.get(tip.name, "unplaced")),\n    ("metric", lambda tip: some_dict[tip.name]),\n]\n'